# Introduction to Ordinary Differential Equations (ODEs)
## A Hands-On Tutorial for Movement Neuroscience Graduate Students

---

**Why this tutorial?**  
Ordinary differential equations (ODEs) are the mathematical language of **dynamics** — they describe how systems change over time. In movement neuroscience, ODEs are everywhere:

- **Muscle models:** the force–velocity–length relationship of Hill-type muscles
- **Limb dynamics:** Newton's equations relating joint torques to angular accelerations
- **Neural models:** integrate-and-fire neurons, population dynamics, and state-space models
- **Motor learning:** error-driven adaptation models (e.g., two-state models of visuomotor adaptation)
- **Postural control:** inverted pendulum models of standing balance

Most of these equations have no closed-form solution, so we solve them **numerically** using algorithms like Euler's method and Runge-Kutta. This notebook teaches you both — from scratch — and then shows you how to use SciPy's professional-grade solver.

**Prerequisites:** Python Basics (L0), NumPy (L1), Matplotlib (L3).  
**Environment:** Google Colab (recommended) or Jupyter Notebook.

---

## Table of Contents

**Part I — Concepts**
1. [What Is an ODE?](#1)
2. [The Key Idea: Turning Derivatives into Steps](#2)

**Part II — Euler's Method**
3. [Euler's Method — The Simplest ODE Solver](#3)
4. [Example 1: Exponential Decay (Passive Muscle Force)](#4)
5. [Euler's Accuracy — The Problem with Large Steps](#5)

**Part III — Runge-Kutta Methods**
6. [Why We Need Better Methods](#6)
7. [The 4th-Order Runge-Kutta Method (RK4)](#7)
8. [Example 2: Spring-Mass-Damper (Limb Dynamics)](#8)
9. [Comparing Euler vs RK4](#9)

**Part IV — SciPy's ODE Solver**
10. [Using `scipy.integrate.solve_ivp`](#10)
11. [Example 3: Inverted Pendulum (Postural Control)](#11)
12. [Example 4: Two-State Motor Adaptation Model](#12)

**Part V — Practice**
13. [Exercises](#13)
14. [Summary & Further Reading](#14)

In [ ]:
# ---- Setup ----

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

---
# Part I — Concepts

## 1. What Is an ODE? <a id='1'></a>

An **ordinary differential equation** describes how a quantity changes over time. It has the general form:

$$\frac{dy}{dt} = f(t, y)$$

This says: "the rate of change of $y$ depends on the current time $t$ and the current value of $y$."

### Movement Neuroscience Examples

| System | ODE | Meaning |
|---|---|---|
| Passive muscle relaxation | $\dot{F} = -F / \tau$ | Force decays exponentially with time constant $\tau$ |
| Mass-spring-damper | $m\ddot{x} + b\dot{x} + kx = F(t)$ | Limb dynamics with mass, damping, and stiffness |
| Leaky integrate-and-fire | $\tau \dot{V} = -(V - V_{rest}) + R \cdot I(t)$ | Membrane voltage driven by input current |
| Inverted pendulum | $\ddot{\theta} = (g/L)\sin\theta - b\dot{\theta} + u(t)$ | Standing balance with gravity and control |
| Motor adaptation | $\dot{x}_s = -x_s/\tau_s + B_s \cdot e$ | Slow state in a two-state adaptation model |

To solve an ODE numerically, we need:
1. The **derivative function** $f(t, y)$
2. An **initial condition** $y(t_0) = y_0$
3. A **time span** to integrate over

---
## 2. The Key Idea: Turning Derivatives into Steps <a id='2'></a>

If we know $y$ at time $t$, and we know the derivative $dy/dt = f(t, y)$, we can **estimate** $y$ at a short time later:

$$y(t + \Delta t) \approx y(t) + f(t, y) \cdot \Delta t$$

This is the fundamental idea behind *all* numerical ODE solvers. The differences between methods (Euler, RK4, etc.) are in **how carefully** they estimate the step.

Think of it like this: you're walking in the dark with a compass that tells you which direction to go *right now*. Euler's method takes one big step in that direction. Runge-Kutta looks ahead multiple times within each step to find a better average direction.

---
# Part II — Euler's Method

## 3. Euler's Method — The Simplest ODE Solver <a id='3'></a>

Euler's method is the simplest possible approach. Given:
- Current state: $y_n$ at time $t_n$
- Step size: $h = \Delta t$

The update rule is:

$$y_{n+1} = y_n + h \cdot f(t_n, y_n)$$

That's it — evaluate the derivative at the current point, multiply by the step size, and add to the current value.

In [ ]:
# ---- Euler's method: general implementation ----

def euler_method(f, y0, t_span, h):
    """
    Solve dy/dt = f(t, y) using Euler's method.
    
    Parameters
    ----------
    f : callable
        The derivative function f(t, y). y can be a scalar or array.
    y0 : float or np.ndarray
        Initial condition.
    t_span : tuple
        (t_start, t_end).
    h : float
        Step size.
    
    Returns
    -------
    t : np.ndarray
        Time points.
    y : np.ndarray
        Solution at each time point.
    """
    t = np.arange(t_span[0], t_span[1] + h, h)
    y0 = np.atleast_1d(np.array(y0, dtype=float))  # Ensure array
    y = np.zeros((len(t), len(y0)))
    y[0] = y0
    
    for i in range(len(t) - 1):
        y[i + 1] = y[i] + h * np.array(f(t[i], y[i]))
    
    return t, y.squeeze()  # squeeze removes extra dimension for scalar ODEs

print("Euler method defined.")

---
## 4. Example 1: Exponential Decay (Passive Muscle Force) <a id='4'></a>

When a muscle relaxes, force decays exponentially:

$$\frac{dF}{dt} = -\frac{F}{\tau}$$

where $\tau$ is the time constant (how quickly force fades). The analytical solution is:

$$F(t) = F_0 \cdot e^{-t/\tau}$$

This is one of the rare ODEs with a closed-form solution, which makes it perfect for testing our numerical solver.

In [ ]:
# ---- Define the ODE: exponential decay ----

tau = 0.5  # Time constant (seconds)
F0 = 100.0 # Initial force (Newtons)

def muscle_decay(t, F):
    """dF/dt = -F/tau — passive muscle force relaxation."""
    return -F / tau

# Analytical (exact) solution for comparison:
def exact_solution(t):
    return F0 * np.exp(-t / tau)

In [ ]:
# ---- Solve with Euler's method ----

h = 0.05  # Step size (seconds)
t_euler, F_euler = euler_method(muscle_decay, F0, (0, 3), h)

# Compare with exact solution
t_exact = np.linspace(0, 3, 500)
F_exact = exact_solution(t_exact)

# Plot
plt.figure(figsize=(10, 5))
plt.plot(t_exact, F_exact, 'k-', linewidth=2, label='Exact solution')
plt.plot(t_euler, F_euler, 'ro--', markersize=4, linewidth=1, label=f'Euler (h={h}s)')
plt.xlabel('Time (s)')
plt.ylabel('Force (N)')
plt.title('Passive Muscle Force Decay')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Compute the error at the final time:
error = abs(F_euler[-1] - exact_solution(t_euler[-1]))
print(f"Final error: {error:.4f} N")

---
## 5. Euler's Accuracy — The Problem with Large Steps <a id='5'></a>

Euler's method is **first-order accurate**: halving the step size halves the error. But it can be dangerously inaccurate with large steps or stiff systems. Let's see the effect of step size:

In [ ]:
# ---- Compare different step sizes ----

step_sizes = [0.5, 0.2, 0.05, 0.01]

plt.figure(figsize=(10, 5))
plt.plot(t_exact, F_exact, 'k-', linewidth=2, label='Exact')

for h in step_sizes:
    t_h, F_h = euler_method(muscle_decay, F0, (0, 3), h)
    error = abs(F_h[-1] - exact_solution(t_h[-1]))
    plt.plot(t_h, F_h, 'o--', markersize=3, label=f'h={h}s (error={error:.3f}N)')

plt.xlabel('Time (s)')
plt.ylabel('Force (N)')
plt.title('Euler Method: Effect of Step Size')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

**Takeaway:** Euler's method works, but you need very small steps for good accuracy. This makes it slow for long simulations or complex systems. We need a better method.

---
# Part III — Runge-Kutta Methods

## 6. Why We Need Better Methods <a id='6'></a>

Euler's method evaluates the derivative **once** per step (at the beginning) and assumes it's constant throughout. That's like driving with your eyes closed — you check the road, then drive straight for a while.

Runge-Kutta methods evaluate the derivative **multiple times** within each step to get a better average. It's like checking the road several times during each stride.

| Method | Derivative evaluations per step | Order of accuracy | Error scales as |
|---|---|---|---|
| Euler | 1 | 1st order | $O(h)$ |
| RK2 (Midpoint) | 2 | 2nd order | $O(h^2)$ |
| **RK4 (Classic)** | **4** | **4th order** | $O(h^4)$ |

RK4 is the workhorse of numerical ODE solving. With 4× the work per step, you get **dramatically** better accuracy.

---
## 7. The 4th-Order Runge-Kutta Method (RK4) <a id='7'></a>

RK4 takes four "trial steps" within each time step and combines them into a weighted average:

$$k_1 = f(t_n,\; y_n)$$
$$k_2 = f\left(t_n + \frac{h}{2},\; y_n + \frac{h}{2} k_1\right)$$
$$k_3 = f\left(t_n + \frac{h}{2},\; y_n + \frac{h}{2} k_2\right)$$
$$k_4 = f(t_n + h,\; y_n + h \cdot k_3)$$

$$y_{n+1} = y_n + \frac{h}{6}(k_1 + 2k_2 + 2k_3 + k_4)$$

Think of it as:
- $k_1$: slope at the **beginning** of the step
- $k_2$: slope at the **midpoint**, using $k_1$ to get there
- $k_3$: slope at the **midpoint** again, using $k_2$ to get there (a better estimate)
- $k_4$: slope at the **end**, using $k_3$ to get there
- Final step: a **weighted average** (midpoint slopes get double weight)

In [ ]:
# ---- RK4: general implementation ----

def rk4_method(f, y0, t_span, h):
    """
    Solve dy/dt = f(t, y) using the classic 4th-order Runge-Kutta method.
    
    Parameters
    ----------
    f : callable
        The derivative function f(t, y).
    y0 : float or np.ndarray
        Initial condition.
    t_span : tuple
        (t_start, t_end).
    h : float
        Step size.
    
    Returns
    -------
    t : np.ndarray
        Time points.
    y : np.ndarray
        Solution at each time point.
    """
    t = np.arange(t_span[0], t_span[1] + h, h)
    y0 = np.atleast_1d(np.array(y0, dtype=float))
    y = np.zeros((len(t), len(y0)))
    y[0] = y0
    
    for i in range(len(t) - 1):
        ti = t[i]
        yi = y[i]
        
        k1 = np.array(f(ti,          yi))
        k2 = np.array(f(ti + h/2,    yi + h/2 * k1))
        k3 = np.array(f(ti + h/2,    yi + h/2 * k2))
        k4 = np.array(f(ti + h,      yi + h   * k3))
        
        y[i + 1] = yi + (h / 6) * (k1 + 2*k2 + 2*k3 + k4)
    
    return t, y.squeeze()

print("RK4 method defined.")

---
## 8. Example 2: Spring-Mass-Damper (Limb Dynamics) <a id='8'></a>

A classic model for limb dynamics is the **spring-mass-damper** system:

$$m\ddot{x} + b\dot{x} + kx = F(t)$$

where $m$ is mass, $b$ is damping, $k$ is stiffness, and $F(t)$ is an external force.

### Converting to a system of first-order ODEs

Most ODE solvers expect **first-order** equations. We convert by defining:
- $y_1 = x$ (position)
- $y_2 = \dot{x}$ (velocity)

Then:
$$\dot{y}_1 = y_2$$
$$\dot{y}_2 = \frac{1}{m}\left[F(t) - b \cdot y_2 - k \cdot y_1\right]$$

This is a **system of 2 coupled ODEs** — a vector-valued problem.

In [ ]:
# ---- Define the spring-mass-damper system ----

m = 1.0    # Mass (kg)
b = 0.5    # Damping coefficient (Ns/m)
k = 4.0    # Spring stiffness (N/m)

def spring_mass_damper(t, y):
    """
    Spring-mass-damper ODE system.
    y[0] = position, y[1] = velocity
    
    Returns [dy0/dt, dy1/dt].
    """
    x, v = y[0], y[1]
    F_ext = 0.0  # No external force (free oscillation)
    
    dxdt = v
    dvdt = (F_ext - b * v - k * x) / m
    
    return [dxdt, dvdt]

# Initial conditions: displaced 5 cm from equilibrium, at rest
y0 = [0.05, 0.0]  # [position (m), velocity (m/s)]
t_span = (0, 10)    # 10 seconds
h = 0.01             # Step size

In [ ]:
# ---- Solve with RK4 ----

t_rk4, y_rk4 = rk4_method(spring_mass_damper, y0, t_span, h)

# Plot position and velocity
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].plot(t_rk4, y_rk4[:, 0] * 100, 'b-', linewidth=1.2)  # Convert m to cm
axes[0].set_ylabel('Position (cm)')
axes[0].set_title('Spring-Mass-Damper: Damped Oscillation (RK4)')
axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_rk4, y_rk4[:, 1] * 100, 'r-', linewidth=1.2)
axes[1].set_ylabel('Velocity (cm/s)')
axes[1].set_xlabel('Time (s)')
axes[1].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 9. Comparing Euler vs RK4 <a id='9'></a>

Let's compare both methods on the same problem to see the difference in accuracy.

In [ ]:
# ---- Compare Euler and RK4 on the muscle decay problem ----

h_test = 0.2  # A deliberately coarse step size to reveal differences

t_euler, F_euler = euler_method(muscle_decay, F0, (0, 3), h_test)
t_rk4, F_rk4 = rk4_method(muscle_decay, F0, (0, 3), h_test)
t_exact = np.linspace(0, 3, 500)
F_exact = exact_solution(t_exact)

plt.figure(figsize=(10, 5))
plt.plot(t_exact, F_exact, 'k-', linewidth=2, label='Exact')
plt.plot(t_euler, F_euler, 'rs--', markersize=6, label=f'Euler (h={h_test}s)')
plt.plot(t_rk4, F_rk4, 'bo--', markersize=6, label=f'RK4 (h={h_test}s)')
plt.xlabel('Time (s)')
plt.ylabel('Force (N)')
plt.title(f'Euler vs RK4 with the SAME step size (h={h_test}s)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

euler_error = abs(F_euler[-1] - exact_solution(t_euler[-1]))
rk4_error = abs(F_rk4[-1] - exact_solution(t_rk4[-1]))
print(f"Final errors with h={h_test}s:")
print(f"  Euler: {euler_error:.4f} N")
print(f"  RK4:   {rk4_error:.6f} N")
print(f"  RK4 is {euler_error/rk4_error:.0f}× more accurate!")

In [ ]:
# ---- Convergence comparison: error vs step size ----

step_sizes = [0.5, 0.2, 0.1, 0.05, 0.02, 0.01]
euler_errors = []
rk4_errors = []

for h in step_sizes:
    _, F_e = euler_method(muscle_decay, F0, (0, 3), h)
    _, F_r = rk4_method(muscle_decay, F0, (0, 3), h)
    t_end = np.arange(0, 3 + h, h)[-1]
    euler_errors.append(abs(F_e[-1] - exact_solution(t_end)))
    rk4_errors.append(abs(F_r[-1] - exact_solution(t_end)))

plt.figure(figsize=(8, 5))
plt.loglog(step_sizes, euler_errors, 'rs-', label='Euler (1st order)')
plt.loglog(step_sizes, rk4_errors, 'bo-', label='RK4 (4th order)')
plt.xlabel('Step size h (s)')
plt.ylabel('Absolute error at t=3s (N)')
plt.title('Convergence: Euler vs RK4')
plt.legend()
plt.grid(True, alpha=0.3, which='both')
plt.show()

**Key insight from the log-log plot:**
- Euler error decreases with slope ≈ 1 (first order: halving $h$ halves error)
- RK4 error decreases with slope ≈ 4 (fourth order: halving $h$ reduces error by 16×)

This is why RK4 is vastly preferred for most practical problems.

---
# Part IV — SciPy's ODE Solver

## 10. Using `scipy.integrate.solve_ivp` <a id='10'></a>

In practice, you'll use SciPy's `solve_ivp` ("solve initial value problem") instead of writing your own solver. It uses **adaptive step sizes** — automatically taking smaller steps when the solution changes rapidly and larger steps when it's smooth.

```python
from scipy.integrate import solve_ivp

sol = solve_ivp(f, t_span, y0, method='RK45', t_eval=t_points)
```

| Parameter | Meaning |
|---|---|
| `f` | Derivative function `f(t, y)` |
| `t_span` | `(t_start, t_end)` |
| `y0` | Initial condition (list or array) |
| `method` | `'RK45'` (default), `'RK23'`, `'DOP853'`, `'Radau'`, `'BDF'` |
| `t_eval` | Specific times at which to report the solution |
| `max_step` | Maximum allowed step size |

The result `sol` has:
- `sol.t` — time points
- `sol.y` — solution (shape: `n_variables × n_timepoints`)
- `sol.success` — whether the solver succeeded

In [ ]:
# ---- Solve the spring-mass-damper with solve_ivp ----

t_eval = np.linspace(0, 10, 1000)  # Times at which we want the solution

sol = solve_ivp(
    spring_mass_damper,    # Derivative function
    (0, 10),               # Time span
    y0,                    # Initial condition [position, velocity]
    method='RK45',         # Adaptive Runge-Kutta (the default)
    t_eval=t_eval,         # Report solution at these times
    max_step=0.01          # Maximum step size
)

print(f"Success: {sol.success}")
print(f"Solution shape: {sol.y.shape}")  # (2, 1000) — 2 variables × 1000 time points

# Plot
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].plot(sol.t, sol.y[0] * 100, 'b-')   # sol.y[0] = position
axes[0].set_ylabel('Position (cm)')
axes[0].set_title('Spring-Mass-Damper solved with scipy.integrate.solve_ivp')
axes[0].grid(True, alpha=0.3)

axes[1].plot(sol.t, sol.y[1] * 100, 'r-')   # sol.y[1] = velocity
axes[1].set_ylabel('Velocity (cm/s)')
axes[1].set_xlabel('Time (s)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 11. Example 3: Inverted Pendulum (Postural Control) <a id='11'></a>

A simple model of standing balance treats the body as an **inverted pendulum** — a rod pivoting at the ankle:

$$\ddot{\theta} = \frac{g}{L}\sin\theta - b\dot{\theta} + u(t)$$

where $\theta$ is the body angle from vertical, $g$ is gravity, $L$ is the effective pendulum length, $b$ is damping, and $u(t)$ is the ankle torque (the "controller").

We'll simulate it with a simple proportional-derivative (PD) controller: $u(t) = -K_p \theta - K_d \dot{\theta}$

In [ ]:
# ---- Inverted pendulum with PD control ----

g = 9.81    # Gravity (m/s²)
L = 1.0     # Pendulum length (m) — roughly COM height
b_pend = 0.1  # Passive damping
Kp = 30.0   # Proportional gain (stiffness of the controller)
Kd = 8.0    # Derivative gain (damping of the controller)

def inverted_pendulum(t, y):
    """
    Inverted pendulum with PD controller.
    y[0] = theta (angle from vertical, rad)
    y[1] = theta_dot (angular velocity, rad/s)
    """
    theta, theta_dot = y[0], y[1]
    
    # PD controller: tries to keep theta = 0
    u = -Kp * theta - Kd * theta_dot
    
    dtheta_dt = theta_dot
    dtheta_dot_dt = (g / L) * np.sin(theta) - b_pend * theta_dot + u
    
    return [dtheta_dt, dtheta_dot_dt]

# Initial condition: tilted 5 degrees forward, at rest
theta0 = np.deg2rad(5)
y0_pend = [theta0, 0.0]

# Solve
t_eval = np.linspace(0, 5, 2000)
sol = solve_ivp(inverted_pendulum, (0, 5), y0_pend, t_eval=t_eval, max_step=0.005)

# Plot
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].plot(sol.t, np.rad2deg(sol.y[0]), 'b-')
axes[0].set_ylabel('Angle (deg)')
axes[0].set_title('Inverted Pendulum with PD Controller (Postural Control)')
axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[0].grid(True, alpha=0.3)

# Compute and plot the control torque
control_torque = -Kp * sol.y[0] - Kd * sol.y[1]
axes[1].plot(sol.t, control_torque, 'r-')
axes[1].set_ylabel('Control Torque (Nm)')
axes[1].set_xlabel('Time (s)')
axes[1].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 12. Example 4: Two-State Motor Adaptation Model <a id='12'></a>

A widely used model in motor learning describes how the brain adapts to a perturbation using two processes — a **fast** process that learns quickly but forgets quickly, and a **slow** process that learns slowly but retains well:

$$\dot{x}_f = -x_f / \tau_f + B_f \cdot e$$
$$\dot{x}_s = -x_s / \tau_s + B_s \cdot e$$

where $e = P - (x_f + x_s)$ is the error (perturbation minus total adaptation), and $P$ is the perturbation.

This model explains why people show **savings** (faster relearning) and **spontaneous recovery** after error-clamp trials.

In [ ]:
# ---- Two-state motor adaptation model ----

tau_f = 5.0     # Fast process time constant (trials)
tau_s = 80.0    # Slow process time constant (trials)
Bf = 0.15       # Fast learning rate
Bs = 0.02       # Slow learning rate

def adaptation_model(t, y, perturbation_func):
    """
    Two-state motor adaptation model.
    y[0] = x_f (fast state)
    y[1] = x_s (slow state)
    """
    xf, xs = y[0], y[1]
    P = perturbation_func(t)        # Current perturbation
    error = P - (xf + xs)           # Motor error
    
    dxf_dt = -xf / tau_f + Bf * error
    dxs_dt = -xs / tau_s + Bs * error
    
    return [dxf_dt, dxs_dt]

# ---- Define the perturbation schedule ----
# Baseline (0-50 trials), 30° rotation (50-200), washout (200-300)

def perturbation(t):
    """Perturbation schedule: 0 → 30 → 0 degrees."""
    if 50 <= t < 200:
        return 30.0   # 30° rotation
    else:
        return 0.0    # No perturbation

# Wrap the ODE to pass the perturbation function
def adaptation_ode(t, y):
    return adaptation_model(t, y, perturbation)

# Solve
t_eval = np.linspace(0, 300, 3000)
sol = solve_ivp(adaptation_ode, (0, 300), [0.0, 0.0], t_eval=t_eval, max_step=0.5)

# Compute derived quantities
xf = sol.y[0]
xs = sol.y[1]
total_adaptation = xf + xs
P_vec = np.array([perturbation(t) for t in sol.t])
error = P_vec - total_adaptation

In [ ]:
# ---- Plot the adaptation model ----

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

# Panel 1: Perturbation and total adaptation
axes[0].fill_between(sol.t, P_vec, alpha=0.2, color='gray', label='Perturbation')
axes[0].plot(sol.t, total_adaptation, 'b-', linewidth=1.5, label='Total adaptation')
axes[0].set_ylabel('Degrees')
axes[0].set_title('Two-State Motor Adaptation Model')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Panel 2: Fast and slow states
axes[1].plot(sol.t, xf, 'r-', label='Fast state ($x_f$)')
axes[1].plot(sol.t, xs, 'g-', label='Slow state ($x_s$)')
axes[1].set_ylabel('Degrees')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

# Panel 3: Motor error
axes[2].plot(sol.t, error, 'k-', linewidth=0.8)
axes[2].set_ylabel('Error (deg)')
axes[2].set_xlabel('Trial')
axes[2].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[2].grid(True, alpha=0.3)

# Mark the phases
for ax in axes:
    ax.axvline(50, color='orange', linestyle=':', alpha=0.7)
    ax.axvline(200, color='orange', linestyle=':', alpha=0.7)

plt.tight_layout()
plt.show()

print("Note: After washout (trial 200), the fast state decays quickly")
print("but the slow state retains — this is 'savings' in motor learning.")

---
# Part V — Practice

## 13. Exercises <a id='13'></a>

---

### Exercise 1: Modify the Muscle Decay Model

The muscle force decay model assumed zero input. Now add a **step input** at $t=1$ s:

$$\frac{dF}{dt} = -\frac{F}{\tau} + I(t)$$

where $I(t) = 50$ for $t \geq 1$ and $I(t) = 0$ otherwise.

1. Modify the derivative function.
2. Solve with RK4 from $t=0$ to $t=5$ with $h=0.01$.
3. Plot the result. What steady-state force does it approach?

In [ ]:
# ---- YOUR CODE HERE ----


### Exercise 2: Spring-Mass-Damper with External Force

Add a **sinusoidal external force** to the spring-mass-damper:

$$F(t) = A \sin(2\pi f \cdot t)$$

with $A = 2$ N and $f = 0.5$ Hz. Use `solve_ivp` to simulate 20 seconds and plot position vs time. What happens when $f$ approaches the natural frequency $\sqrt{k/m} / (2\pi)$?

In [ ]:
# ---- YOUR CODE HERE ----


### Exercise 3: Explore the Adaptation Model Parameters

Using the two-state adaptation model from Example 4:
1. What happens if you make $\tau_s$ much larger (e.g., 200)? Simulate and plot.
2. What happens if $B_f = 0$ (no fast learning)?
3. Add an **error-clamp** phase (trials 200–250) where error is forced to 0, followed by a second exposure to the perturbation (trials 250–350). Do you see savings?

In [ ]:
# ---- YOUR CODE HERE ----


### Exercise 4: Lotka-Volterra (Challenge)

The Lotka-Volterra equations model predator-prey dynamics (or competing neural populations):

$$\dot{x} = \alpha x - \beta x y$$
$$\dot{y} = \delta x y - \gamma y$$

with $\alpha=1.1$, $\beta=0.4$, $\delta=0.1$, $\gamma=0.4$.

1. Implement the derivative function.
2. Solve with `solve_ivp` from $t=0$ to $t=50$ with $y_0 = [10, 5]$.
3. Plot both populations vs time.
4. Plot a **phase portrait** ($y$ vs $x$).

In [ ]:
# ---- YOUR CODE HERE ----


---
## 14. Summary & Further Reading <a id='14'></a>

### What You Learned

| Concept | What it does | When to use |
|---|---|---|
| **ODE** | $dy/dt = f(t, y)$ — describes how a system evolves | Modeling any dynamic system |
| **Euler's method** | $y_{n+1} = y_n + h \cdot f(t_n, y_n)$ | Teaching/understanding; quick prototyping |
| **RK4** | 4 derivative evaluations per step, 4th-order accuracy | Manual solver when you need control |
| **`solve_ivp`** | SciPy's adaptive solver (RK45 default) | Production code — always use this |
| **System of ODEs** | Convert $\ddot{x}$ to two first-order equations | Any 2nd-order system (most mechanics) |
| **Phase portrait** | Plot $y$ vs $x$ instead of vs time | Visualize system dynamics |

### The Recipe for Solving Any ODE Numerically

1. **Write the derivative function** `f(t, y)` → returns `dy/dt`
2. **Convert higher-order ODEs** to first-order systems ($\ddot{x} \to [x, \dot{x}]$)
3. **Set initial conditions** $y_0$
4. **Call `solve_ivp`** with your function, time span, and initial conditions
5. **Plot and analyze** `sol.t` and `sol.y`

### Further Reading

- [SciPy `solve_ivp` documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html)
- [Strogatz — *Nonlinear Dynamics and Chaos*](https://www.stevenstrogatz.com/books/nonlinear-dynamics-and-chaos-with-applications-to-physics-biology-chemistry-and-engineering) — the gold standard textbook
- [Shadmehr & Wise — *Computational Neurobiology of Reaching and Pointing*](https://www.shadmehrlab.org/book) — motor control models using ODEs
- [Smith, Ghazizadeh & Shadmehr (2006)](https://doi.org/10.1371/journal.pbio.0040179) — the two-state adaptation model
- [Winter — *Biomechanics and Motor Control of Human Movement*](https://www.wiley.com/en-us/Biomechanics+and+Motor+Control+of+Human+Movement%2C+4th+Edition-p-9780470398180) — equations of motion for limb dynamics

### What's Next?

With ODE solving mastered, you can:
- **Simulate Hill-type muscle models** with activation dynamics
- **Build forward dynamics simulations** of reaching movements
- **Implement optimal control** models (LQR, iLQG) that plan movements by solving ODEs
- **Fit adaptation models** to real behavioral data using parameter optimization

---

*Happy simulating, and may your solutions always converge!*